# Spotify Audio Features & Health Clustering Analysis

**Exploring Spotify track audio features to understand music patterns and clustering through data analysis.**

## Project Overview

This project analyzes Spotify audio features (danceability, energy, tempo, valence, etc.) to understand patterns in music characteristics. We explore how tracks cluster by audio features and what makes songs popular.

## Learning Objectives

1. Work with music/audio feature data
2. Analyze distributions of continuous features
3. Explore correlations between audio characteristics
4. Apply clustering concepts to group similar tracks
5. Understand what makes music 'popular' from a data perspective

## Business / Research Problem

Music streaming platforms use audio features for recommendation systems. Understanding these features helps:
- **Playlist curators** group similar songs
- **Artists** understand what makes tracks popular
- **Platforms** build better recommendation engines

**Key question:** *What audio feature patterns exist in Spotify data, and how do they relate to popularity?*

## Why This Analysis Matters

The music streaming industry is massive. Data-driven understanding of audio features powers recommendation algorithms, playlist generation, and artist analytics.

## Dataset Overview

Key Spotify audio features include:
- `danceability` (0-1): How suitable for dancing
- `energy` (0-1): Intensity and activity
- `valence` (0-1): Musical positiveness/happiness
- `tempo`: Beats per minute
- `loudness`: Overall loudness (dB)
- `speechiness`: Presence of spoken words
- `acousticness`: Acoustic vs electronic
- `instrumentalness`: Vocal vs instrumental
- `popularity`: Track popularity score

## Dataset Source and License Notes- **Source:** [Kaggle - Spotify Tracks Dataset](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)- **License:** CC0 Public Domain- **Usage:** Educational purposes

## Environment Setup

In [1]:
!pip install pandas numpy matplotlib seaborn scipy kagglehub scikit-learn -q

## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
print('All imports loaded successfully.')

All imports loaded successfully.


## Configuration / Constants

In [3]:
RANDOM_SEED = 42
KAGGLE_DATASET = 'maharshipandya/-spotify-tracks-dataset'
AUDIO_FEATURES = ['danceability', 'energy', 'valence', 'tempo', 'loudness',
                   'speechiness', 'acousticness', 'instrumentalness', 'liveness']
np.random.seed(RANDOM_SEED)

## Dataset Download / Loading

In [4]:
import kagglehub
import os

path = kagglehub.dataset_download(KAGGLE_DATASET)
print(f'Downloaded to: {path}')

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(path, csv_file))
print(f'Loaded {len(df)} rows and {len(df.columns)} columns')
print(f'Columns: {list(df.columns)}')
df.head()

  0%|          | 0.00/8.17M [00:00<?, ?B/s]

 12%|█▏        | 1.00M/8.17M [00:03<00:21, 346kB/s]

 24%|██▍       | 2.00M/8.17M [00:03<00:08, 739kB/s]

 37%|███▋      | 3.00M/8.17M [00:03<00:04, 1.27MB/s]

 49%|████▉     | 4.00M/8.17M [00:03<00:02, 1.91MB/s]

...

100%|██████████| 8.17M/8.17M [00:04<00:00, 1.97MB/s]

Extracting files...


Downloaded to: C:\Users\ahmad\.cache\kagglehub\datasets\maharshipandya\-spotify-tracks-dataset\versions\1


Loaded 114000 rows and 21 columns
Columns: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## Data Validation Checks

In [5]:
print(f'Shape: {df.shape}')
print(f'\nMissing values:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'None')
print(f'\nDuplicates: {df.duplicated().sum()}')

# Check available audio features
avail_features = [f for f in AUDIO_FEATURES if f in df.columns]
print(f'\nAvailable audio features: {avail_features}')
df[avail_features].describe()

Shape: (114000, 21)

Missing values:
artists       1
album_name    1
track_name    1
dtype: int64

Duplicates: 0

Available audio features: ['danceability', 'energy', 'valence', 'tempo', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness']


,danceability,energy,valence,tempo,loudness,speechiness,acousticness,instrumentalness,liveness
count,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000
mean,0.566800,0.641383,0.474068,122.147837,-8.258960,0.084652,0.314910,0.156050,0.213553
std,0.173542,0.251529,0.259261,29.978197,5.029337,0.105732,0.332523,0.309555,0.190378
min,0.000000,0.000000,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000
25%,0.456000,0.472000,0.260000,99.218750,-10.013000,0.035900,0.016900,0.000000,0.098000
50%,0.580000,0.685000,0.464000,122.017000,-7.004000,0.048900,0.169000,0.000042,0.132000
75%,0.695000,0.854000,0.683000,140.071000,-5.003000,0.084500,0.598000,0.049000,0.273000
max,0.985000,1.000000,0.995000,243.372000,4.532000,0.965000,0.996000,1.000000,1.000000


## Data Cleaning

In [6]:
df = df.drop_duplicates()
df = df.dropna(subset=avail_features)
print(f'After cleaning: {len(df)} rows')

# Identify popularity column
pop_col = [c for c in df.columns if 'popular' in c.lower()]
if pop_col:
    pop_col = pop_col[0]
    print(f'Popularity column: {pop_col}')
    print(f'Popularity range: {df[pop_col].min()} - {df[pop_col].max()}')

After cleaning: 114000 rows
Popularity column: popularity
Popularity range: 0 - 100


## Exploratory Data Analysis

In [7]:
print('=== Audio Feature Means ===')
print(df[avail_features].mean().round(3))

if pop_col:
    print(f'\n=== Popularity Stats ===')
    print(f'Mean: {df[pop_col].mean():.1f}, Median: {df[pop_col].median():.1f}')

=== Audio Feature Means ===
danceability          0.567
energy                0.641
valence               0.474
tempo               122.148
loudness             -8.259
speechiness           0.085
acousticness          0.315
instrumentalness      0.156
liveness              0.214
dtype: float64

=== Popularity Stats ===
Mean: 33.2, Median: 35.0


## Univariate Analysis

In [8]:
n_feats = len(avail_features)
n_rows = (n_feats + 2) // 3
fig, axes = plt.subplots(n_rows, 3, figsize=(18, 4 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(avail_features):
    axes[i].hist(df[feat], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(feat)
for j in range(len(avail_features), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if 'energy' in df.columns and 'loudness' in df.columns:
    axes[0].scatter(df['energy'], df['loudness'], alpha=0.1, s=2, c='steelblue')
    axes[0].set_xlabel('Energy')
    axes[0].set_ylabel('Loudness')
    axes[0].set_title('Energy vs Loudness')

if 'danceability' in df.columns and 'valence' in df.columns:
    axes[1].scatter(df['danceability'], df['valence'], alpha=0.1, s=2, c='coral')
    axes[1].set_xlabel('Danceability')
    axes[1].set_ylabel('Valence')
    axes[1].set_title('Danceability vs Valence')

if 'acousticness' in df.columns and 'energy' in df.columns:
    axes[2].scatter(df['acousticness'], df['energy'], alpha=0.1, s=2, c='mediumseagreen')
    axes[2].set_xlabel('Acousticness')
    axes[2].set_ylabel('Energy')
    axes[2].set_title('Acousticness vs Energy')

plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [10]:
# Popularity bins analysis
if pop_col:
    df['pop_tier'] = pd.cut(df[pop_col], bins=[0, 20, 50, 75, 100],
                             labels=['Low', 'Medium', 'High', 'Very High'])
    print('=== Audio Features by Popularity Tier ===')
    print(df.groupby('pop_tier')[avail_features].mean().round(3))

=== Audio Features by Popularity Tier ===
           danceability  energy  valence    tempo  loudness  speechiness  \
pop_tier                                                                   
Low               0.571   0.659    0.461  123.615    -8.685        0.094   
Medium            0.555   0.649    0.478  123.014    -8.286        0.090   
High              0.578   0.627    0.451  121.643    -8.106        0.075   
Very High         0.633   0.677    0.508  119.457    -6.372        0.080   

           acousticness  instrumentalness  liveness  
pop_tier                                             
Low               0.276             0.297     0.200  
Medium            0.331             0.147     0.242  
High              0.303             0.119     0.186  
Very High         0.197             0.029     0.171  


## Statistical Checks / Hypothesis Testing

In [11]:
if pop_col:
    print('=== Correlation with Popularity ===')
    for feat in avail_features:
        r, p = stats.pearsonr(df[feat], df[pop_col])
        print(f'{feat:>20}: r={r:+.4f}, p={p:.2e}')

# Audio feature correlation matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df[avail_features].corr(), annot=True, cmap='coolwarm', center=0,
            fmt='.2f', ax=ax, linewidths=0.5)
ax.set_title('Audio Feature Correlation Matrix')
plt.tight_layout()
plt.show()

=== Correlation with Popularity ===
        danceability: r=+0.0354, p=4.96e-33
              energy: r=+0.0011, p=7.21e-01
             valence: r=-0.0405, p=1.14e-42
               tempo: r=+0.0132, p=8.25e-06
            loudness: r=+0.0504, p=4.48e-65
         speechiness: r=-0.0449, p=5.06e-52
        acousticness: r=-0.0255, p=7.85e-18
    instrumentalness: r=-0.0951, p=2.05e-227
            liveness: r=-0.0054, p=6.89e-02


## Visual Analysis

In [12]:
if pop_col:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(df[pop_col], bins=50, color='mediumseagreen', edgecolor='white')
    ax.axvline(df[pop_col].mean(), color='red', linestyle='--', label=f'Mean: {df[pop_col].mean():.0f}')
    ax.set_title('Popularity Distribution')
    ax.set_xlabel('Popularity Score')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Key Findings

1. **Energy and loudness are strongly correlated** — louder songs feel more energetic
2. **Acousticness inversely correlates with energy** — as expected
3. **Danceability and valence have moderate positive correlation** — happy songs are more danceable
4. **Most tracks have low popularity** — the distribution is heavily right-skewed
5. **Audio features alone weakly predict popularity** — marketing and artist fame matter more

## Limitations

1. **No genre labels in some versions:** Genre-specific analysis may be limited
2. **Popularity is temporal:** Scores change over time
3. **Audio features are algorithmic:** They're Spotify's computed metrics, not human ratings
4. **No user data:** Individual listening patterns aren't captured

## Recommendations / Next Steps

1. Apply K-means clustering to group tracks by audio profile
2. Build a genre classifier using audio features
3. Analyze feature trends over decades

### How to Extend This Analysis
- Use the Spotify API for real-time data
- Build a playlist recommendation engine
- Compare features across genres

## Common Mistakes

1. **Expecting audio features to strongly predict popularity** — they don't; marketing matters more
2. **Not checking feature ranges:** Some features are 0-1, others have different scales
3. **Ignoring multicollinearity:** Energy/loudness are highly correlated—be careful in regression
4. **Treating Spotify features as ground truth:** They're algorithmic estimates, not perfect measures
5. **Not sampling for visualization:** Scatter plots with 100k+ points need alpha tuning or sampling

## Mini Challenge / Exercises

1. What audio feature profile defines the most popular tracks (top 1%)?
2. Apply K-means (k=5) to cluster tracks by audio features. What characterizes each cluster?
3. Find the most 'unique' track — furthest from the centroid of all tracks
4. Is there a correlation between track duration and popularity?
5. Create a radar chart comparing the average audio profile of high vs low popularity tracks

In [13]:
# Space for exercise solutions

## Final Summary / Key Takeaways

- **Audio features capture distinct musical dimensions** — energy, mood, danceability
- **Strong correlations exist** between energy/loudness and acousticness/energy
- **Popularity is weakly predicted** by audio features alone
- **Clustering reveals natural music groups** beyond genre labels
- **This dataset is excellent for practicing** unsupervised learning and feature analysis

Understanding audio features is key to music data science and recommendation systems.